# Multiclass Classification Pipeline for DCombo

This notebook trains and evaluates multiple classification models for predicting DCombo using scikit-learn pipelines with stratified cross-validation.

## Requirements
- Python 3.8+
- scikit-learn
- pandas
- numpy
- xgboost (optional)

## Usage
1. Ensure `X_train.csv` and `Y_train.csv` are in the same directory as this notebook
2. Run all cells in sequence
3. The notebook will generate the following output files:
   - `labels_mapping.csv` - Mapping of class labels to indices
   - `cv_results_<model>.csv` - Cross-validation metrics for each model
   - `confusion_matrix.csv` - Confusion matrix for the best model
   - `classification_report.csv` - Classification report for the best model
   - `feature_importances.csv` - Feature importances (if applicable)

## Models
The script trains and compares the following models:
1. **RandomForestClassifier** - Ensemble of decision trees
2. **XGBClassifier** - Gradient boosting (if xgboost is installed)
3. **MultinomialNB** - Naive Bayes classifier

The best model is selected based on F1-macro score.

## Key Features
- Automatic detection of numeric and categorical features
- Proper preprocessing pipeline with ColumnTransformer
- Stratified 6-fold cross-validation
- Hyperparameter tuning with RandomizedSearchCV
- Comprehensive evaluation metrics (accuracy, F1-macro, precision, recall)
- No data leakage - all preprocessing inside pipelines

## 1. Imports and Setup

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    StratifiedKFold, cross_validate, cross_val_predict, RandomizedSearchCV
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    confusion_matrix, classification_report, make_scorer,
    accuracy_score, f1_score, precision_score, recall_score
)

# Try to import XGBoost
XGBOOST_AVAILABLE = False
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("✓ XGBoost is available")
except ImportError:
    print("✗ XGBoost not available, will skip XGBClassifier")

# Set random state for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Load Data

In [ ]:
# Load training data
print("Loading data...")
X_train = pd.read_csv('X_train.csv')
Y_train_df = pd.read_csv('Y_train.csv')
y_train = Y_train_df['DCombo'].astype(str)  # Ensure categorical/string type

print(f"✓ Data loaded: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"✓ Target classes: {sorted(y_train.unique())}")
print(f"\nClass distribution:")
print(y_train.value_counts().sort_index())

## 3. Helper Functions

In [ ]:
def build_preprocessor(X):
    """
    Build a ColumnTransformer for preprocessing numeric and categorical features.
    
    Parameters:
    -----------
    X : DataFrame or array-like
        Training features
    
    Returns:
    --------
    ColumnTransformer
        Preprocessor with numeric and categorical transformers
    """
    if isinstance(X, pd.DataFrame):
        # Detect numeric and categorical columns automatically
        numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
        categorical_features = X.select_dtypes(include=['object', 'category']).columns.tolist()
    else:
        # If not DataFrame, treat all as numeric
        n_features = X.shape[1] if hasattr(X, 'shape') else len(X[0])
        numeric_features = [f"feature_{i}" for i in range(n_features)]
        categorical_features = []
    
    print(f"Numeric features: {len(numeric_features)}")
    print(f"Categorical features: {len(categorical_features)}")
    
    # Build transformers
    transformers = []
    
    if numeric_features:
        numeric_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler(with_mean=False))  # with_mean=False for OHE compatibility
        ])
        transformers.append(('num', numeric_transformer, numeric_features))
    
    if categorical_features:
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True))
        ])
        transformers.append(('cat', categorical_transformer, categorical_features))
    
    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder='drop'
    )
    
    return preprocessor

In [ ]:
def build_models(preprocessor):
    """
    Build dictionary of models with their pipelines and hyperparameter grids.
    
    Parameters:
    -----------
    preprocessor : ColumnTransformer
        Fitted preprocessor
    
    Returns:
    --------
    dict
        Dictionary with model names as keys and (pipeline, param_grid, needs_label_encoding) tuples as values
    """
    models = {}
    
    # Random Forest
    rf_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=RANDOM_STATE))
    ])
    rf_param_grid = {
        'classifier__n_estimators': [50, 100, 200, 300],
        'classifier__max_depth': [None, 5, 10, 15, 20],
        'classifier__min_samples_split': [2, 5, 10],
        'classifier__min_samples_leaf': [1, 2, 4],
        'classifier__max_features': ['sqrt', 'log2', None]
    }
    models['random_forest'] = (rf_pipeline, rf_param_grid, False)  # No label encoding needed
    
    # XGBoost (if available) - needs label encoding
    if XGBOOST_AVAILABLE:
        xgb_pipeline = Pipeline([
            ('preprocessor', preprocessor),
            ('classifier', XGBClassifier(random_state=RANDOM_STATE, eval_metric='mlogloss'))
        ])
        xgb_param_grid = {
            'classifier__n_estimators': [50, 100, 200, 300],
            'classifier__max_depth': [3, 5, 7, 9],
            'classifier__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'classifier__subsample': [0.6, 0.8, 1.0],
            'classifier__colsample_bytree': [0.6, 0.8, 1.0]
        }
        models['xgb'] = (xgb_pipeline, xgb_param_grid, True)  # Needs label encoding
    
    # Multinomial Naive Bayes
    # Note: MultinomialNB requires non-negative features
    # We'll use it with the preprocessor which outputs sparse matrix from OHE
    nb_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', MultinomialNB())
    ])
    nb_param_grid = {
        'classifier__alpha': [0.01, 0.1, 0.5, 1.0, 2.0, 5.0]
    }
    models['nb'] = (nb_pipeline, nb_param_grid, False)  # No label encoding needed
    
    return models

In [ ]:
def evaluate_with_cv(model, X, y, cv, scoring):
    """
    Evaluate model using cross-validation.
    
    Parameters:
    -----------
    model : Pipeline
        Model pipeline to evaluate
    X : DataFrame or array-like
        Features
    y : Series or array-like
        Target variable
    cv : cross-validation generator
        Cross-validation strategy
    scoring : dict
        Scoring metrics
    
    Returns:
    --------
    dict
        Cross-validation results
    """
    cv_results = cross_validate(
        model, X, y,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )
    return cv_results

In [ ]:
def fit_and_report(model_name, pipeline, param_grid, X, y, cv, scoring, needs_label_encoding=False):
    """
    Fit model with hyperparameter tuning and generate reports.
    
    Parameters:
    -----------
    model_name : str
        Name of the model
    pipeline : Pipeline
        Model pipeline
    param_grid : dict
        Hyperparameter grid
    X : DataFrame or array-like
        Features
    y : Series or array-like
        Target variable
    cv : cross-validation generator
        Cross-validation strategy
    scoring : dict
        Scoring metrics
    needs_label_encoding : bool
        Whether to encode labels to integers
    
    Returns:
    --------
    tuple
        (best_pipeline, cv_results_df, mean_f1_macro, label_encoder)
    """
    print(f"\n{'='*60}")
    print(f"Training {model_name}...")
    print(f"{'='*60}")
    
    # Handle label encoding if needed
    label_encoder = None
    y_encoded = y
    if needs_label_encoding:
        print("Encoding labels for XGBoost...")
        label_encoder = LabelEncoder()
        y_encoded = label_encoder.fit_transform(y)
    
    # Hyperparameter tuning with RandomizedSearchCV
    print("Performing hyperparameter search...")
    search = RandomizedSearchCV(
        pipeline,
        param_distributions=param_grid,
        n_iter=20,
        cv=cv,
        scoring='f1_macro',
        n_jobs=-1,
        random_state=RANDOM_STATE,
        error_score='raise'
    )
    
    try:
        search.fit(X, y_encoded)
        best_pipeline = search.best_estimator_
        
        print(f"✓ Best parameters: {search.best_params_}")
        print(f"✓ Best f1_macro score: {search.best_score_:.4f}")
        
        # Evaluate best model with cross_validate to get detailed metrics
        print("Evaluating best model with cross-validation...")
        cv_results = evaluate_with_cv(best_pipeline, X, y_encoded, cv, scoring)
        
        # Create results DataFrame
        results_data = {
            'fold': list(range(1, len(cv_results['test_accuracy']) + 1)),
            'accuracy': cv_results['test_accuracy'],
            'f1_macro': cv_results['test_f1_macro'],
            'precision_macro': cv_results['test_precision_macro'],
            'recall_macro': cv_results['test_recall_macro']
        }
        cv_results_df = pd.DataFrame(results_data)
        
        # Print summary
        print("\nCross-validation results:")
        print(cv_results_df)
        print("\nMean scores:")
        print(f"  Accuracy:  {cv_results_df['accuracy'].mean():.4f} ± {cv_results_df['accuracy'].std():.4f}")
        print(f"  F1-macro:  {cv_results_df['f1_macro'].mean():.4f} ± {cv_results_df['f1_macro'].std():.4f}")
        print(f"  Precision: {cv_results_df['precision_macro'].mean():.4f} ± {cv_results_df['precision_macro'].std():.4f}")
        print(f"  Recall:    {cv_results_df['recall_macro'].mean():.4f} ± {cv_results_df['recall_macro'].std():.4f}")
        
        # Save results
        output_file = f"cv_results_{model_name}.csv"
        cv_results_df.to_csv(output_file, index=False)
        print(f"\n✓ Results saved to {output_file}")
        
        mean_f1 = cv_results_df['f1_macro'].mean()
        return best_pipeline, cv_results_df, mean_f1, label_encoder
        
    except Exception as e:
        print(f"✗ Error training {model_name}: {str(e)}")
        return None, None, 0.0, None

In [ ]:
def create_labels_mapping(y):
    """
    Create and save labels mapping table.
    
    Parameters:
    -----------
    y : Series or array-like
        Target variable
    """
    # Get unique classes and their counts
    classes = np.unique(y)
    value_counts = pd.Series(y).value_counts()
    
    # Create mapping DataFrame
    labels_df = pd.DataFrame({
        'label_index': range(len(classes)),
        'label_name': classes,
        'support': [value_counts[c] for c in classes]
    })
    
    # Save to CSV
    labels_df.to_csv('labels_mapping.csv', index=False)
    print("\nLabels mapping:")
    print(labels_df)
    print("\n✓ Labels mapping saved to labels_mapping.csv")

In [ ]:
def generate_best_model_reports(best_pipeline, model_name, X, y, cv, label_encoder=None):
    """
    Generate confusion matrix, classification report, and feature importances for best model.
    
    Parameters:
    -----------
    best_pipeline : Pipeline
        Best trained pipeline
    model_name : str
        Name of the best model
    X : DataFrame or array-like
        Features
    y : Series or array-like
        Target variable
    cv : cross-validation generator
        Cross-validation strategy
    label_encoder : LabelEncoder, optional
        Label encoder if used for this model
    """
    print(f"\n{'='*60}")
    print(f"Generating reports for best model: {model_name}")
    print(f"{'='*60}")
    
    # Encode labels if needed
    y_for_model = y
    if label_encoder is not None:
        y_for_model = label_encoder.transform(y)
    
    # Get cross-validated predictions
    print("Generating cross-validated predictions...")
    y_pred = cross_val_predict(best_pipeline, X, y_for_model, cv=cv, method='predict')
    
    # Decode predictions if needed
    if label_encoder is not None:
        y_pred = label_encoder.inverse_transform(y_pred)
    
    # Confusion Matrix
    print("\nCreating confusion matrix...")
    cm = confusion_matrix(y, y_pred)
    classes = sorted(np.unique(y))
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)
    cm_df.to_csv('confusion_matrix.csv')
    print("Confusion Matrix:")
    print(cm_df)
    print("✓ Confusion matrix saved to confusion_matrix.csv")
    
    # Classification Report
    print("\nCreating classification report...")
    report_dict = classification_report(y, y_pred, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv('classification_report.csv')
    print("Classification Report:")
    print(report_df)
    print("✓ Classification report saved to classification_report.csv")
    
    # Feature Importances (if available)
    print("\nChecking for feature importances...")
    classifier = best_pipeline.named_steps['classifier']
    
    if hasattr(classifier, 'feature_importances_'):
        print("Extracting feature importances...")
        
        # Get feature names from preprocessor
        preprocessor = best_pipeline.named_steps['preprocessor']
        feature_names = preprocessor.get_feature_names_out()
        
        # Create importances DataFrame
        importances_df = pd.DataFrame({
            'feature': feature_names,
            'importance': classifier.feature_importances_
        })
        importances_df = importances_df.sort_values('importance', ascending=False)
        importances_df.to_csv('feature_importances.csv', index=False)
        
        print("\nTop 20 Feature Importances:")
        print(importances_df.head(20))
        print("✓ Feature importances saved to feature_importances.csv")
    elif hasattr(classifier, 'coef_'):
        print("Model has coefficients but not feature importances.")
        print("(Skipping feature importance export for this model type)")
    else:
        print("Model does not support feature importances.")

## 4. Build Preprocessor

In [ ]:
print("\nBuilding preprocessor...")
preprocessor = build_preprocessor(X_train)
print("✓ Preprocessor built successfully")

## 5. Define Cross-Validation Strategy and Scoring Metrics

In [ ]:
# Stratified K-Fold CV
cv = StratifiedKFold(n_splits=6, shuffle=True, random_state=RANDOM_STATE)
print(f"Cross-validation: {cv.n_splits}-fold stratified")

# Scoring metrics
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'f1_macro': make_scorer(f1_score, average='macro'),
    'precision_macro': make_scorer(precision_score, average='macro', zero_division=0),
    'recall_macro': make_scorer(recall_score, average='macro', zero_division=0)
}
print("Metrics: accuracy, f1_macro, precision_macro, recall_macro")

## 6. Build and Train Models

In [ ]:
print("\nBuilding models...")
models = build_models(preprocessor)
print(f"✓ {len(models)} models prepared: {list(models.keys())}")

In [ ]:
# Train and evaluate all models
results = {}

for model_name, (pipeline, param_grid, needs_encoding) in models.items():
    best_pipe, cv_results_df, mean_f1, label_enc = fit_and_report(
        model_name, pipeline, param_grid, X_train, y_train, cv, scoring, needs_encoding
    )
    if best_pipe is not None:
        results[model_name] = {
            'pipeline': best_pipe,
            'cv_results': cv_results_df,
            'mean_f1_macro': mean_f1,
            'label_encoder': label_enc
        }

## 7. Compare Models and Select Best

In [ ]:
print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)

# Create comparison DataFrame
comparison_data = []
for model_name, result in results.items():
    cv_df = result['cv_results']
    comparison_data.append({
        'Model': model_name,
        'Accuracy': f"{cv_df['accuracy'].mean():.4f} ± {cv_df['accuracy'].std():.4f}",
        'F1-Macro': f"{cv_df['f1_macro'].mean():.4f} ± {cv_df['f1_macro'].std():.4f}",
        'Precision': f"{cv_df['precision_macro'].mean():.4f} ± {cv_df['precision_macro'].std():.4f}",
        'Recall': f"{cv_df['recall_macro'].mean():.4f} ± {cv_df['recall_macro'].std():.4f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print("\n", comparison_df.to_string(index=False))

# Select best model based on f1_macro
best_model_name = max(results.items(), key=lambda x: x[1]['mean_f1_macro'])[0]
best_pipeline = results[best_model_name]['pipeline']
best_f1 = results[best_model_name]['mean_f1_macro']
best_label_encoder = results[best_model_name]['label_encoder']

print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model_name}")
print(f"F1-Macro Score: {best_f1:.4f}")
print(f"{'='*60}")

## 8. Generate Labels Mapping

In [ ]:
create_labels_mapping(y_train)

## 9. Generate Detailed Reports for Best Model

In [ ]:
generate_best_model_reports(best_pipeline, best_model_name, X_train, y_train, cv, best_label_encoder)

## 10. Summary

In [ ]:
print("\n" + "="*60)
print("EXECUTION COMPLETE")
print("="*60)
print("\nGenerated files:")
print("  - labels_mapping.csv")
for model_name in results.keys():
    print(f"  - cv_results_{model_name}.csv")
print("  - confusion_matrix.csv (best model)")
print("  - classification_report.csv (best model)")
if hasattr(best_pipeline.named_steps['classifier'], 'feature_importances_'):
    print("  - feature_importances.csv (best model)")
print("\n✓ All tasks completed successfully!")